
## Example 1: Multi-Agent Collaboration Using Sequential Execution in CrewAI

### Objective
Demonstrate how to create a **multi-agent system** using CrewAI, where two specialized AI agents collaborate to answer a student question.

The example combines **Retrieval-Augmented Generation (RAG)** with **sequential task execution** to compare Breadth-First Search (BFS) and A* Search using information from Artificial Intelligence course materials.

The system consists of two agents:
- **Agent 1 – Course Researcher:** Retrieves relevant information from the course document using RAG.
- **Agent 2 – Educational Explainer:** Uses the research findings to generate a clear, student-friendly explanation.

### Implementation

1. **Load environment variables:** Use `load_dotenv()` to load API credentials from the `.env` file.
   - Verify that `GEMINI_API_KEY` is available.
   - Raise an error if the API key is missing.

2. **Initialize the LLM:** Configure a Google Gemini model with:
   - `temperature=0.1`: Produces more deterministic responses.
   - `max_tokens=1000`: Limits the maximum number of generated tokens per LLM response.

3. **Initialize the RAG tool:** Configure `RagTool` to retrieve information from course materials.
   - `model_name="all-MiniLM-L6-v2"`: Specifies the embedding model.
   - `device="cpu"`: Executes embedding computations on the CPU.
   - `normalize_embeddings=True`: Normalizes the generated vector embeddings.
   - `limit=5`: Sets the default maximum number of retrieved results.
   - `similarity_threshold=0.25`: Specifies the similarity threshold.
   - `collection_name`: Identifies the vector collection.

4. **Index the document:** Use `rag_tool.add()` to process `AI_Course_Extended.txt`.
   - Split the document into chunks.
   - Generate vector embeddings.
   - Store the embeddings in a vector database.

5. **Define the student question:** Compare BFS and A* regarding frontier ordering, heuristic functions, optimality, complexity, and efficiency.

6. **Create two specialized agents:**
   - **Course Researcher:** Uses `RagTool` to retrieve relevant information from the course document.
   - **Educational Explainer:** Uses the LLM to transform research findings into an understandable explanation.
   - `allow_delegation=False`: Prevents agents from delegating tasks.
   - `max_iter`: Limits the number of execution iterations for each agent.
   - `verbose=True`: Displays detailed execution logs.

7. **Define two tasks:**
   - **Task 1 – Research:** Retrieve and summarize 3 to 5 relevant findings from the course document.
   - **Task 2 – Explanation:** Use the research findings to produce a structured explanation, including a comparison table and an example.

8. **Establish task dependencies:** Use `context=[research_task]` to explicitly provide the output of Task 1 as context for Task 2.

9. **Create a sequential Crew:** Use `Process.sequential` to execute the tasks in their defined order.

10. **Execute the Crew:** Use `await sequential_crew.kickoff_async()` to execute the workflow asynchronously in Jupyter Notebook.

11. **Display the results:** Print the research summary, educational explanation, and final Crew output.

### Expected Output

**Task 1 – Course Researcher**
- Retrieves relevant document chunks using RAG.
- Identifies important information about BFS and A*.
- Produces a structured research summary.

**Task 2 – Educational Explainer**
- Receives the research summary from Task 1.
- Synthesizes the information into a coherent explanation.
- Generates a comparison table.
- Includes a practical example for students.

**Final Crew Result**
- Returns the final output produced by the Educational Explainer.

### Key Concepts
- Multi-agent systems
- Agent specialization (roles, goals, and backstories)
- Retrieval-Augmented Generation (RAG)
- Task definition and assignment
- Sequential execution
- Task dependencies and context sharing
- Agent collaboration
- Asynchronous execution

### Important Concept: Sequential Execution

In CrewAI, `Process.sequential` executes tasks in the order specified in the Crew configuration.

**Execution Flow:**

1. **Task 1 → Course Researcher**
   - Uses RAG to retrieve relevant information.
   - Produces a research summary.

2. **Task 2 → Educational Explainer**
   - Receives the research summary.
   - Generates the final explanation.

The second task begins only after the first task has completed.

### Important Concept: Task Context

The following parameter establishes an explicit dependency between tasks:

`context=[research_task]`

This means that the Educational Explainer receives the output of the research task as additional context.

**Important observations:**
- The developer defines the tasks and their execution order.
- The developer assigns each task to a specific agent.
- The output of one task can be provided as context to another task.
- Agents do not automatically decide to change the execution order.
- Only the Course Researcher has direct access to `RagTool`.
- The Educational Explainer uses the retrieved research findings without directly accessing the vector database.

### Key Takeaway

**Sequential multi-agent execution allows complex problems to be divided into smaller tasks performed by specialized agents.**

In this example, the Course Researcher retrieves domain-specific knowledge, while the Educational Explainer transforms that knowledge into a student-friendly answer.

By combining **RAG, agent specialization, and task dependencies**, CrewAI supports structured workflows in which multiple agents collaborate toward a common objective.


In [2]:

import os
from pathlib import Path
from dotenv import load_dotenv

from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import RagTool


# ==================================================
# 1. LOAD ENVIRONMENT VARIABLES
# ==================================================

load_dotenv(override=True)

if not os.getenv("GEMINI_API_KEY"):
    raise ValueError("GEMINI_API_KEY is missing from .env")


# ==================================================
# 2. CONFIGURE THE LLM
# ==================================================

llm = LLM(
    model="gemini/gemini-3.5-flash-lite",
    temperature=0.1,
    max_tokens=1000
)


# ==================================================
# 3. INITIALIZE THE RAG TOOL
# ==================================================

rag_tool = RagTool(
    config={
        "embedding_model": {
            "provider": "sentence-transformer",
            "config": {
                "model_name": "all-MiniLM-L6-v2",
                "device": "cpu",
                "normalize_embeddings": True
            }
        }
    },
    summarize=False,
    limit=5,
    similarity_threshold=0.25,
    collection_name="ai_course_sequential_demo"
)

print("RAG tool initialized successfully!")


# ==================================================
# 4. ADD DOCUMENT TO KNOWLEDGE BASE
# ==================================================

document_path = Path("files/AI_Course_Extended.txt").resolve()

if not document_path.is_file():
    raise FileNotFoundError(
        f"Document not found: {document_path}"
    )

rag_tool.add(
    data_type="file",
    path=str(document_path)
)

print("Document indexed successfully!")


# ==================================================
# 5. DEFINE THE STUDENT QUESTION
# ==================================================

question = (
    "Compare BFS and A* search algorithms regarding "
    "frontier ordering, heuristic functions, optimality, "
    "time and space complexity, and practical efficiency."
)


# ==================================================
# 6. CREATE TWO AGENTS
# ==================================================

research_agent = Agent(
    role="Course Researcher",
    goal=(
        "Retrieve accurate information from the "
        "Artificial Intelligence course materials."
    ),
    backstory=(
        "You are an experienced academic researcher "
        "who specializes in retrieving relevant "
        "information from technical documents."
    ),
    tools=[rag_tool],
    llm=llm,
    allow_delegation=False,
    max_iter=4,
    verbose=True
)


explanation_agent = Agent(
    role="Educational Explainer",
    goal=(
        "Transform retrieved technical information "
        "into clear explanations for students."
    ),
    backstory=(
        "You are an experienced university lecturer "
        "who explains Artificial Intelligence "
        "concepts using simple examples."
    ),
    llm=llm,
    allow_delegation=False,
    max_iter=3,
    verbose=True
)


# ==================================================
# 7. DEFINE TWO TASKS
# ==================================================

research_task = Task(
    description=(
        "Research the following question:\n"
        "{question}\n\n"
        "Use RagTool to retrieve relevant passages "
        "from the AI course document. "
        "Identify approximately 3 to 5 important "
        "passages or findings. "
        "Do not invent information or sources."
    ),
    expected_output=(
        "A structured research summary containing "
        "3 to 5 relevant findings from the document."
    ),
    agent=research_agent
)


explanation_task = Task(
    description=(
        "Answer the student question:\n"
        "{question}\n\n"
        "Use the research findings from Task 1. "
        "Combine and reformulate the information "
        "into a coherent student-friendly explanation. "
        "Include a comparison table and an example. "
        "Do not add unsupported information."
    ),
    expected_output=(
        "A clear explanation comparing BFS and A*, "
        "including a table and a practical example."
    ),
    agent=explanation_agent,
    context=[research_task]
)


# ==================================================
# 8. CREATE SEQUENTIAL CREW
# ==================================================

sequential_crew = Crew(
    agents=[
        research_agent,
        explanation_agent
    ],
    tasks=[
        research_task,
        explanation_task
    ],
    process=Process.sequential,
    verbose=True
)


# ==================================================
# 9. EXECUTE CREW (JUPYTER)
# ==================================================

sequential_result = await sequential_crew.kickoff_async(
    inputs={"question": question}
)


# ==================================================
# 10. DISPLAY BOTH TASK OUTPUTS
# ==================================================

print("\n===== TASK 1: RESEARCH =====\n")
print(research_task.output.raw)

print("\n===== TASK 2: EXPLANATION =====\n")
print(explanation_task.output.raw)

print("\n===== FINAL CREW RESULT =====\n")
print(sequential_result.raw)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAG tool initialized successfully!
Document indexed successfully!


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.27                                                                                       │
│  Latest version:  1.15.30                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6cf50661-b81a-4812-888f-09bd4a4415b9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the following question:                                                                         │
│  Compare BFS and A* search algorithms regarding frontier ordering, heuristic functions, optimality, time and    │
│  space complexity, and practical efficiency.                                                                    │
│                                                                                                                 │
│  Use RagTool to retrieve relevant passages from the AI course document. Identify approximately 3 to 5           │
│  important passages or findings. Do not invent information or sources.                                          │
│  ID: 11e4af7c-fde0-47fd-99ea-8e2154bc916e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Educational Explainer                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Welcome back, everyone! Today, we are going to look at two fundamental search algorithms in Artificial         │
│  Intelligence: **Breadth-First Search (BFS)** and **A\* Search**.                                               │
│                                                                                                                 │
│  Imagine you are trying to find a path through a maze. How do you decide which way to turn? Do you              │
│  systematically check every path of length 1, then every path of length 2, and so on? Or do you use a sense of  │
│  direction—a hunch—to guide you toward the exit?                                                                │
│                                                                                                                 │
│  BFS and A* represent these two different philosophies. Let’s break down how they compare across several key    │
│  categories.                                                                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Frontier Ordering and Evaluation Keys                                                                   │
│  The "frontier" is the set of all candidate paths currently discovered but not yet fully explored. How these    │
│  algorithms organize their frontiers changes everything:                                                        │
│  * **BFS:** Uses a simple **FIFO (First-In, First-Out) queue**—just like a standard waiting line. It explores   │
│  shallower nodes before deeper ones, completely ignoring any sense of direction or cost.                        │
│  * **A\*:** Uses a **priority queue** ordered by an evaluation function, $f = g + h$.                           │
│    * $g$ is the cumulative past cost to get from the start to the current node.                                 │
│    * $h$ is the heuristic estimate of the remaining cost from the current node to the goal.                     │
│                                                                                                                 │
│  ### 2. Heuristic Functions                                                                                     │
│  * **BFS:** Requires **no heuristic**. It operates purely on unweighted steps or depth.                         │
│  * **A\*:** Relies heavily on a **heuristic function ($h$)**. To guarantee we find the absolute best path,      │
│  this heuristic must be *admissible*, meaning it must never overestimate the true remaining cost. If $h$ is     │
│  set to zero, A* behaves just like Uniform Cost Search. If $h$ is weak or misleading, A* loses its efficiency   │
│  advantage.                                                                                                     │
│                                                                                                                 │
│  ### 3. Optimality                                                                                              │
│  * **BFS:** Is only optimal for shortest paths when **e

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer the student question:                                                                             │
│  Compare BFS and A* search algorithms regarding frontier ordering, heuristic functions, optimality, time and    │
│  space complexity, and practical efficiency.                                                                    │
│                                                                                                                 │
│  Use the research findings from Task 1. Combine and reformulate the information into a coherent                 │
│  student-friendly explanation. Include a comparison table and an example. Do not add unsupported information.   │
│  Agent: Educational Explainer                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6cf50661-b81a-4812-888f-09bd4a4415b9                                                                       │
│  Final Output: Welcome back, everyone! Today, we are going to look at two fundamental search algorithms in      │
│  Artificial Intelligence: **Breadth-First Search (BFS)** and **A\* Search**.                                    │
│                                                                                                                 │
│  Imagine you are trying to find a path through a maze. How do you decide which way to turn? Do you              │
│  systematically check every path of length 1, then every path of length 2, and so on? Or do you use a sense of  │
│  direction—a hunch—to guide you toward the exit?                                                                │
│                                                                                                                 │
│  BFS and A* represent these two different philosophies. Let’s break down how they compare across several key    │
│  categories.                                                                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Frontier Ordering and Evaluation Keys                                                                   │
│  The "frontier" is the set of all candidate paths currently discovered but not yet fully explored. How these    │
│  algorithms organize their frontiers changes everything:                                                        │
│  * **BFS:** Uses a simple **FIFO (First-In, First-Out) queue**—just like a standard waiting line. It explores   │
│  shallower nodes before deeper ones, completely ignoring any sense of direction or cost.                        │
│  * **A\*:** Uses a **priority queue** ordered by an evaluation function, $f = g + h$.                           │
│    * $g$ is the cumulative past cost to get from the start to the current node.                                 │
│    * $h$ is the heuristic estimate of the remaining cost from the current node to the goal.                     │
│                                                                                                                 │
│  ### 2. Heuristic Functions                                                                                     │
│  * **BFS:** Requires **no heuristic**. It operates purely on unweighted steps or depth.                         │
│  * **A\*:** Relies heavily on a **heuristic function ($h$)**. To guarantee we find the absolute best path,      │
│  this heuristic must be *admissible*, meaning it must never overestimate the true remaining cost. If $h$ is     │
│  set to zero, A* behaves just like Uniform Cost Search. If $h$ is weak or misleading, A* loses its efficiency   │
│  advantage.                                                                                                     │
│                                                                                                                 │
│  ### 3. Optimality                                                                                              │
│  * **BFS:** Is only optimal for shortest paths when **


===== TASK 1: RESEARCH =====

Based on the AI course materials retrieved via the knowledge base, here is a structured comparison of the BFS and A* search algorithms:

1. **Frontier Ordering and Evaluation Keys**
   - **BFS:** Uses an ordinary FIFO (First-In, First-Out) queue. It prioritizes nodes based on their depth or number of actions/edges, expanding shallower nodes before deeper ones. It is uninformed and does not use a heuristic.
   - **A/A\***: Uses a priority queue ordered by an evaluation function combining known past costs and estimated future costs ($f = g + h$), where $g$ is the cumulative cost from the start and $h$ is the heuristic estimate of the remaining cost to the goal.

2. **Heuristic Functions**
   - **BFS:** Requires no heuristic. It evaluates paths purely based on the number of uniform steps or depth.
   - **A\*:** Requires a heuristic function ($h$) to estimate the cost from a given node to the goal. For A* to guarantee optimality, the heuristic must be admissi

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## Example 2: Multi-Agent Collaboration Using Hierarchical Execution in CrewAI

### Objective

Demonstrate how to create a **hierarchical multi-agent system** using CrewAI, where a manager agent coordinates specialized agents to accomplish a common objective.

The example generates a **university micro-credential proposal** for an Artificial Intelligence course worth 5 ECTS credits.

Unlike sequential execution, where tasks are executed in a predefined order, hierarchical execution introduces a **manager agent responsible for coordinating work, delegating responsibilities, and integrating results**.

The system consists of three agents:
- **Agent 1 – Curriculum Specialist:** Designs learning outcomes and course topics.
- **Agent 2 – Assessment Specialist:** Proposes assessment methods and evaluation criteria.
- **Manager Agent – Academic Program Manager:** Coordinates the specialists and integrates their contributions into a complete proposal.

### Implementation

1. **Load environment variables:** Use `load_dotenv()` to load API credentials from the `.env` file.

2. **Initialize the LLM:** Configure a Google Gemini model with:
   - `temperature=0.1`: Produces more deterministic responses.
   - `max_tokens=1500`: Limits the maximum number of generated tokens per LLM response.

3. **Create two specialized agents:**
   - **Curriculum Specialist:** Responsible for designing learning outcomes and course content.
   - **Assessment Specialist:** Responsible for defining assessment methods and evaluation criteria.
   - `role`: Defines each agent's specialization.
   - `goal`: Specifies what each agent aims to achieve.
   - `backstory`: Provides context about each agent's expertise.
   - `allow_delegation=False`: Prevents the specialists from delegating tasks to other agents.
   - `verbose=True`: Displays detailed execution information.

4. **Create the manager agent:** Define an Academic Program Manager responsible for coordinating the specialists.
   - `role="Academic Program Manager"`: Defines the management responsibility.
   - `goal`: Specifies the overall objective of producing a micro-credential proposal.
   - `backstory`: Describes the manager's coordination responsibilities.
   - `allow_delegation=True`: Allows the manager to delegate work.
   - `max_iter=10`: Limits the manager's execution iterations.
   - `verbose=True`: Displays the manager's execution details.

5. **Define the task:** Instruct the system to generate a micro-credential proposal for a university course.
   - The micro-credential must have 5 ECTS credits.
   - Define four learning outcomes.
   - Identify four main course topics.
   - Propose assessment methods and evaluation criteria.
   - Integrate the contributions into one structured proposal.

6. **Create a hierarchical Crew:** Configure the multi-agent system using:
   - `agents`: Lists the two specialized agents.
   - `tasks`: Contains the overall proposal generation task.
   - `manager_agent`: Specifies the agent responsible for coordination.
   - `process=Process.hierarchical`: Enables manager-led execution.
   - `verbose=True`: Displays detailed Crew execution logs.

7. **Execute the Crew:** Use `await crew.kickoff_async()` to start the workflow asynchronously, providing `course_name="Artificial Intelligence"` as input.

8. **Measure execution time:** Use `time.perf_counter()` to calculate the total duration of the multi-agent execution.

9. **Display the final result:** Use `result.raw` to display the generated micro-credential proposal.

### Expected Output

The system generates a structured micro-credential proposal containing:

**1. General Information**
- Micro-credential title: Artificial Intelligence
- Workload: 5 ECTS credits

**2. Learning Outcomes**
- Four measurable learning outcomes describing what students should know or be able to demonstrate.

**3. Course Content**
- Four main topics related to Artificial Intelligence.

**4. Assessment**
- Proposed assessment methods.
- Evaluation criteria for measuring student achievement.

**5. Integrated Proposal**
- A coherent document combining curriculum design and assessment recommendations.

### Key Concepts
- Hierarchical multi-agent systems
- Manager and worker agents
- Agent specialization
- Task delegation
- Manager-led coordination
- Dynamic assignment of work
- Result integration
- Asynchronous execution
- Execution time measurement

### Important Concept: Hierarchical Execution

In CrewAI, `Process.hierarchical` introduces a manager responsible for coordinating the execution of tasks.

Unlike sequential execution, the manager can decide how to distribute work among the available specialized agents.

**Execution Flow:**

1. **Manager Agent – Academic Program Manager**
   - Receives the micro-credential generation task.
   - Analyzes the requirements.
   - Determines how to coordinate the specialists.

2. **Curriculum Specialist**
   - Can be delegated responsibility for developing learning outcomes and course topics.
   - Returns its findings to the manager.

3. **Assessment Specialist**
   - Can be delegated responsibility for proposing assessment methods and evaluation criteria.
   - Returns its findings to the manager.

4. **Manager Agent – Final Integration**
   - Reviews the specialists' contributions.
   - Combines the results.
   - Produces the final micro-credential proposal.

**Important:** Delegation decisions are made by the manager using the LLM. The manager is instructed to involve both specialists, but the exact sequence and number of delegations are not guaranteed.

### Important Concept: Manager vs. Worker Agents

**Manager Agent**
- Coordinates the overall workflow.
- Decides which specialist should perform delegated work.
- Reviews and integrates specialist contributions.
- Can request additional work when necessary.
- Is responsible for producing the final result.

**Worker Agents**
- Perform specialized work assigned by the manager.
- Operate according to their roles, goals, and backstories.
- Return results to the manager.
- Do not independently delegate work when `allow_delegation=False`.

### Important Observation: Sequential vs. Hierarchical Execution

**Example 1: Sequential Execution**
- The developer defines two tasks.
- Each task is assigned to a specific agent.
- Tasks are executed in a predefined order.
- The output of Task 1 is provided to Task 2 using `context=[research_task]`.
- Coordination is primarily determined by the developer.

**Example 2: Hierarchical Execution**
- The developer defines one overall task.
- The manager coordinates the specialized agents.
- The manager decides how to delegate work during execution.
- Specialists return their contributions to the manager.
- The manager integrates the results into a final answer.

### Important Observation: Task Delegation

In this example, the following configuration enables hierarchical coordination:

`process=Process.hierarchical`

`manager_agent=manager_agent`

The manager is configured with:

`allow_delegation=True`

This allows the manager to assign work to specialized agents during execution.

However, **hierarchical execution does not guarantee that every specialist will be used**.

The actual delegation decisions depend on the manager's LLM reasoning and the task requirements.

Use `verbose=True` to inspect which agents participated and what work was delegated.

### Key Takeaway

**Hierarchical multi-agent execution introduces a manager agent that coordinates specialized agents and dynamically delegates work toward a common objective.**

In this example, the Academic Program Manager coordinates curriculum design and assessment planning to generate an Artificial Intelligence micro-credential proposal.

Compared with sequential execution, hierarchical execution provides greater flexibility in task delegation and coordination, but it may require additional LLM calls and execution time.

**Important:** The generated micro-credential is a proposal based on the LLM's knowledge and task instructions. Because no university documents or external tools are provided, its contents should be reviewed against official course materials and institutional requirements.


In [3]:

import time

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

# ==========================================
# 1. LOAD ENVIRONMENT VARIABLES
# ==========================================

load_dotenv()

# ==========================================
# 2. CONFIGURE THE LLM
# ==========================================

llm = LLM(
    model="gemini/gemini-3.5-flash-lite",
    temperature=0.1,
    max_tokens=1500
)

# ==========================================
# 3. CREATE SPECIALIZED AGENTS
# ==========================================

# Agent 1: Curriculum Specialist
curriculum_agent = Agent(
    role="University Curriculum Specialist",
    goal="Design learning outcomes and course content",
    backstory=(
        "You are a university professor experienced "
        "in curriculum design and AI education."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False
)

# Agent 2: Assessment Specialist
assessment_agent = Agent(
    role="University Assessment Specialist",
    goal="Design appropriate student assessment methods",
    backstory=(
        "You are an academic specialist experienced "
        "in student evaluation and assessment design."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False
)

# ==========================================
# 4. CREATE THE MANAGER AGENT
# ==========================================

manager_agent = Agent(
    role="Academic Program Manager",
    goal=(
        "Coordinate specialists to create a complete "
        "university micro-credential proposal"
    ),
    backstory=(
        "You are an experienced academic coordinator. "
        "You delegate curriculum work to the Curriculum "
        "Specialist and assessment work to the Assessment "
        "Specialist. You review and integrate their results."
    ),
    llm=llm,
    allow_delegation=True,
    max_iter=10,
    verbose=True
)

# ==========================================
# 5. DEFINE THE TASK
# ==========================================

proposal_task = Task(
    description=(
        "Generate a micro-credential proposal based on "
        "the university course: {course_name}. "
        "The micro-credential has 5 ECTS credits. "
        "Coordinate the specialists to define: "
        "1. Four learning outcomes, "
        "2. Four main course topics, "
        "3. Assessment methods and criteria. "
        "Delegate curriculum design to the Curriculum "
        "Specialist and assessment design to the "
        "Assessment Specialist. "
        "Integrate their contributions into one proposal."
    ),
    expected_output=(
        "A structured micro-credential proposal containing "
        "the title, 4 learning outcomes, 4 course topics, "
        "and assessment methods with evaluation criteria."
    )
)

# ==========================================
# 6. CREATE HIERARCHICAL CREW
# ==========================================

crew = Crew(
    agents=[
        curriculum_agent,
        assessment_agent
    ],
    tasks=[proposal_task],
    manager_agent=manager_agent,
    process=Process.hierarchical,
    verbose=True
)

# ==========================================
# 7. EXECUTE THE CREW
# ==========================================

start = time.perf_counter()

try:
    result = await crew.kickoff_async(
        inputs={
            "course_name": "Artificial Intelligence"
        }
    )

    print("\n===== MICRO-CREDENTIAL PROPOSAL =====\n")
    print(result.raw)

except Exception as e:
    print("Error:", e)

finally:
    elapsed = time.perf_counter() - start
    print(f"\nExecution time: {elapsed:.2f} seconds")


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.27                                                                                       │
│  Latest version:  1.15.30                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c9e04254-50a1-4801-b916-9b46385d71bb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Generate a micro-credential proposal based on the university course: Artificial Intelligence. The        │
│  micro-credential has 5 ECTS credits. Coordinate the specialists to define: 1. Four learning outcomes, 2. Four  │
│  main course topics, 3. Assessment methods and criteria. Delegate curriculum design to the Curriculum           │
│  Specialist and assessment design to the Assessment Specialist. Integrate their contributions into one          │
│  proposal.                                                                                                      │
│  ID: bbc0b3dd-fe1e-4677-ba2f-fdbb6c93252d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Academic Program Manager                                                                                │
│                                                                                                                 │
│  Task: Generate a micro-credential proposal based on the university course: Artificial Intelligence. The        │
│  micro-credential has 5 ECTS credits. Coordinate the specialists to define: 1. Four learning outcomes, 2. Four  │
│  main course topics, 3. Assessment methods and criteria. Delegate curriculum design to the Curriculum           │
│  Specialist and assessment design to the Assessment Specialist. Integrate their contributions into one          │
│  proposal.                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'coworker': 'University Curriculum Specialist', 'context': "We need to develop a micro-credential       │
│  proposal based on the university course 'Artificial Intelligence' carrying 5 ECTS credits. The curric...       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Academic Program Manager                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # University Micro-Credential Proposal                                                                         │
│                                                                                                                 │
│  **Micro-Credential Title:** Applied Artificial Intelligence: Fundamentals, Machine Learning, and Responsible   │
│  Practice                                                                                                       │
│  **Credit Value:** 5 ECTS (approx. 125–150 student workload hours)                                              │
│  **Level:** Undergraduate / Continuing Professional Development                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Micro-Credential Overview                                                                                │
│  This 5 ECTS micro-credential provides learners with a comprehensive, hands-on introduction to Artificial       │
│  Intelligence. Designed to bridge foundational theory with practical application, the program equips            │
│  participants with the skills to understand core AI paradigms, implement supervised and unsupervised machine    │
│  learning models, leverage modern generative AI tools effectively, and navigate the ethical and governance      │
│  challenges inherent in deploying AI systems.                                                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2. Learning Outcomes (LOs)                                                                                  │
│  *Upon successful completion of this 5 ECTS micro-credential, students will be able to:*                        │
│                                                                                                                 │
│  1. **LO1: Comprehend** fundamental AI concepts, historical evolution, and core paradigms (including search     │
│  algorithms, knowledge representation, and machine learning) to critically evaluate the scope and limitations   │
│  of current AI technologies.                                                                                    │
│  2. **LO2: Apply** supervised and unsupervised machine learning algorithms using standard industry tools        │
│  (e.g., Python, Scikit-Learn) to solve well-defined classification, regression, and clustering problems.        │
│  3. **LO3: Utilize** modern generative AI models and prompt engineering techniques to enhance productivity,     │
│  content creation, and problem-solving workflows across various domains.                                        │
│  4. **LO4: Analyze** the ethical, legal, and societal implications of AI deployment—such as bias, privacy,      │
│  transparency, and accountability—and propose mitigatio

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Generate a micro-credential proposal based on the university course: Artificial Intelligence. The        │
│  micro-credential has 5 ECTS credits. Coordinate the specialists to define: 1. Four learning outcomes, 2. Four  │
│  main course topics, 3. Assessment methods and criteria. Delegate curriculum design to the Curriculum           │
│  Specialist and assessment design to the Assessment Specialist. Integrate their contributions into one          │
│  proposal.                                                                                                      │
│  Agent: Academic Program Manager                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== MICRO-CREDENTIAL PROPOSAL =====

# University Micro-Credential Proposal

**Micro-Credential Title:** Applied Artificial Intelligence: Fundamentals, Machine Learning, and Responsible Practice  
**Credit Value:** 5 ECTS (approx. 125–150 student workload hours)  
**Level:** Undergraduate / Continuing Professional Development  

---

## 1. Micro-Credential Overview
This 5 ECTS micro-credential provides learners with a comprehensive, hands-on introduction to Artificial Intelligence. Designed to bridge foundational theory with practical application, the program equips participants with the skills to understand core AI paradigms, implement supervised and unsupervised machine learning models, leverage modern generative AI tools effectively, and navigate the ethical and governance challenges inherent in deploying AI systems.

---

## 2. Learning Outcomes (LOs)
*Upon successful completion of this 5 ECTS micro-credential, students will be able to:*

1. **LO1: Comprehend** fundamental AI co

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: c9e04254-50a1-4801-b916-9b46385d71bb                                                                       │
│  Final Output: # University Micro-Credential Proposal                                                           │
│                                                                                                                 │
│  **Micro-Credential Title:** Applied Artificial Intelligence: Fundamentals, Machine Learning, and Responsible   │
│  Practice                                                                                                       │
│  **Credit Value:** 5 ECTS (approx. 125–150 student workload hours)                                              │
│  **Level:** Undergraduate / Continuing Professional Development                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Micro-Credential Overview                                                                                │
│  This 5 ECTS micro-credential provides learners with a comprehensive, hands-on introduction to Artificial       │
│  Intelligence. Designed to bridge foundational theory with practical application, the program equips            │
│  participants with the skills to understand core AI paradigms, implement supervised and unsupervised machine    │
│  learning models, leverage modern generative AI tools effectively, and navigate the ethical and governance      │
│  challenges inherent in deploying AI systems.                                                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2. Learning Outcomes (LOs)                                                                                  │
│  *Upon successful completion of this 5 ECTS micro-credential, students will be able to:*                        │
│                                                                                                                 │
│  1. **LO1: Comprehend** fundamental AI concepts, historical evolution, and core paradigms (including search     │
│  algorithms, knowledge representation, and machine learning) to critically evaluate the scope and limitations   │
│  of current AI technologies.                                                                                    │
│  2. **LO2: Apply** supervised and unsupervised machine learning algorithms using standard industry tools        │
│  (e.g., Python, Scikit-Learn) to solve well-defined classification, regression, and clustering problems.        │
│  3. **LO3: Utilize** modern generative AI models and prompt engineering techniques to enhance productivity,     │
│  content creation, and problem-solving workflows across various domains.                                        │
│  4. **LO4: Analyze** the ethical, legal, and societal implications of AI deployment—such as bias, privacy,      │
│  transparency, and accountability—and propose mitigati

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## Example 3: Hierarchical Multi-Agent Execution with Automatic Manager and Task Dependencies

### Objective

Demonstrate how to create a **hierarchical multi-agent system** using CrewAI, where an automatically created manager coordinates specialized agents to complete multiple related tasks.

The example uses two specialized agents to explain important concepts of the **Transformer architecture** and generate a beginner-friendly educational lesson.

Unlike Example 2, where a custom manager agent was explicitly defined, this example uses `manager_llm` to automatically create the manager.

The system consists of:
- **Agent 1 – AI Researcher:** Specializes in explaining Transformer architecture and deep learning concepts.
- **Agent 2 – Educational Writer:** Specializes in transforming technical information into beginner-friendly educational materials.
- **Automatic Manager:** Coordinates task execution, assigns tasks to appropriate agents, and reviews their results.

### Implementation

1. **Load environment variables:** Use `load_dotenv()` to load API credentials from the `.env` file.

2. **Initialize the LLM:** Configure a Google Gemini model with:
   - `model="gemini/gemini-3.5-flash-lite"`: Specifies the language model.
   - `temperature=0.1`: Produces more deterministic responses.

3. **Create two specialized agents:**
   - **AI Researcher:** Responsible for explaining concepts related to Transformers and deep learning.
   - **Educational Writer:** Responsible for preparing beginner-friendly explanations.
   - `role`: Defines the agent's specialization.
   - `goal`: Specifies what the agent aims to accomplish.
   - `backstory`: Provides context about the agent's expertise.
   - `llm`: Specifies the language model used by the agent.
   - `verbose=True`: Displays detailed execution information.

4. **Define two tasks:**

   **Task 1 – Research**
   - Explain three important concepts of Transformer architecture.
   - Produce a structured explanation of the selected concepts.

   **Task 2 – Educational Writing**
   - Create a beginner-friendly lesson about Transformers.
   - Use the results produced by Task 1.
   - Generate a short educational explanation.

5. **Define task dependencies:** Use `context=[task1]` in Task 2.
   - Makes the output of Task 1 available as context for Task 2.
   - Allows the second task to build upon the findings of the first task.
   - Establishes an explicit information dependency between the tasks.

6. **Create a hierarchical Crew:** Configure the multi-agent system using:
   - `agents=[researcher, writer]`: Specifies the available specialized agents.
   - `tasks=[task1, task2]`: Defines the tasks to execute.
   - `process=Process.hierarchical`: Enables hierarchical execution.
   - `manager_llm=llm`: Automatically creates a manager using the specified LLM.
   - `verbose=True`: Displays detailed execution logs.

7. **Execute the Crew:** Use `await crew.kickoff_async()` to start the workflow asynchronously in Jupyter Notebook.

8. **Inspect the results:** Display the outputs generated for each task using:
   - `task1.output.raw`: Research findings.
   - `task2.output.raw`: Educational explanation.

### Expected Output

**Task 1 – Transformer Research**
- Three important concepts related to Transformer architecture.
- Technical explanations of the selected concepts.
- Information that can be used to prepare educational materials.

**Task 2 – Educational Lesson**
- A beginner-friendly explanation of Transformer architecture.
- Concepts reformulated using accessible terminology.
- A coherent lesson based on the findings from Task 1.

### Key Concepts
- Hierarchical multi-agent systems
- Automatic manager creation
- Manager-led coordination
- Specialized worker agents
- Dynamic task assignment
- Task dependencies
- Context sharing between tasks
- LLM-based task execution
- Asynchronous execution

### Important Concept: Automatic Manager Creation

CrewAI supports two approaches for configuring a manager in hierarchical execution.

**Approach 1 – Explicit Manager Agent (Example 2)**

The developer creates a dedicated manager agent and provides it through:

`manager_agent=manager_agent`

This approach allows the developer to customize the manager's role, goal, backstory, and other parameters.

**Approach 2 – Automatic Manager (Example 3)**

The developer provides only the manager's language model:

`manager_llm=llm`

CrewAI automatically creates a manager responsible for coordinating the available worker agents.

The manager uses the LLM to decide which available agent is most appropriate for executing each task.

### Important Concept: Dynamic Task Assignment

In this example, the tasks are defined without explicitly specifying an agent:

`task1 = Task(...)`

`task2 = Task(...)`

Therefore, the manager determines which available worker agent should execute each task.

**Possible Assignment:**

1. **Task 1 → AI Researcher**
   - The manager selects the researcher because the task requires technical knowledge about Transformers.

2. **Task 2 → Educational Writer**
   - The manager selects the writer because the task requires educational content generation.

**Important:** These assignments are not explicitly defined in the code. The manager makes the decisions during execution, so the actual assignments may differ.

### Important Concept: Task Dependencies

Task 2 contains the following parameter:

`context=[task1]`

This creates an explicit dependency between the tasks.

**Execution Flow:**

1. Task 1 is executed and produces a research summary.
2. The output of Task 1 becomes available as context for Task 2.
3. Task 2 uses this information to generate an educational explanation.

The dependency ensures that Task 2 can use the information produced by Task 1.

### Important Observation: Task Order vs. Agent Assignment

Hierarchical execution provides flexibility in assigning tasks to agents, but this does not mean the manager can arbitrarily change task dependencies.

In this example:

`tasks=[task1, task2]`

The tasks are defined in a specific order.

Additionally:

`context=[task1]`

Task 2 explicitly depends on the output of Task 1.

Therefore, the expected execution is:

**Task 1 → Task 2**

The manager decides which agent executes each task, rather than freely reversing the defined task dependencies.

### Important Observation: Example 2 vs. Example 3

**Example 2: Hierarchical Execution with Explicit Manager**
- A custom manager agent is defined.
- Two specialized agents are available.
- One overall task is defined.
- The manager coordinates the specialists to generate a micro-credential proposal.
- The manager's role, goal, and backstory are explicitly configured.

**Example 3: Hierarchical Execution with Automatic Manager**
- The manager is created automatically using `manager_llm`.
- Two specialized agents are available.
- Two separate tasks are defined.
- Task 2 explicitly depends on Task 1.
- The manager dynamically assigns tasks to available agents.

### Key Takeaway

**Hierarchical execution allows a manager to coordinate specialized agents and decide which agent should perform each task.**

In this example, CrewAI automatically creates the manager using `manager_llm`, while the developer defines two related tasks with an explicit context dependency.

The example illustrates the difference between:
- **Task order:** Defined by the workflow and its dependencies.
- **Agent assignment:** Determined dynamically by the manager.
- **Task context:** Information transferred between related tasks.

**Important:** This example does not use RAG or external search tools. The agents rely on the language model's existing knowledge and the information shared between tasks.


In [5]:

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

# 1. Load API key
load_dotenv()

# 2. Configure LLM
llm = LLM(
    model="gemini/gemini-3.5-flash-lite",
    temperature=0.1
)

# 3. Define worker agents
researcher = Agent(
    role="AI Researcher",
    goal="Explain AI concepts accurately",
    backstory="Expert in Transformers and deep learning.",
    llm=llm,
    verbose=True
)

writer = Agent(
    role="Educational Writer",
    goal="Create beginner-friendly educational lessons",
    backstory="Experienced university educator.",
    llm=llm,
    verbose=True
)

# 4. Define tasks
task1 = Task(
    description="Explain three concepts of Transformer architecture.",
    expected_output="Three important concepts."
)

task2 = Task(
    description="Write a beginner-friendly lesson about Transformers.",
    expected_output="A short educational explanation.",
    context=[task1]
)

# 5. Configure hierarchical crew
crew = Crew(
    agents=[researcher, writer],
    tasks=[task1, task2],
    process=Process.hierarchical,
    manager_llm=llm,
    verbose=True
)

result = await crew.kickoff_async()

# 7. Inspect results
print("\nTASK 1 OUTPUT:")
print(task1.output.raw)

print("\nTASK 2 OUTPUT:")
print(task2.output.raw)


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.27                                                                                       │
│  Latest version:  1.15.30                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5a1e0ac1-64f9-4b1a-81bd-dc6734b376dd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Explain three concepts of Transformer architecture.                                                      │
│  ID: b0e6be79-9115-41a7-83d0-49a8c2105612                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Explain three concepts of Transformer architecture.                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'context': 'The user wants an explanation of three concepts of Transformer architecture. The criteria   │
│  is that three important concepts must be explained. The final answer must be the actual complete ...           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Demystifying Transformers: A Beginner-Friendly Guide                                                         │
│                                                                                                                 │
│  Welcome to the engine room of modern Artificial Intelligence! If you’ve used tools like ChatGPT, Claude, or    │
│  advanced language translators, you have interacted with a piece of architecture called the **Transformer**.    │
│                                                                                                                 │
│  Introduced in 2017 by Google researchers in the landmark paper *"Attention Is All You Need,"* Transformers     │
│  revolutionized how computers understand human language.                                                        │
│                                                                                                                 │
│  Before Transformers, AI processed text like a conveyor belt—word by word, sequentially. Transformers changed   │
│  everything by looking at an entire sentence all at once. But how do they manage to understand context, word    │
│  order, and multiple meanings simultaneously?                                                                   │
│                                                                                                                 │
│  Let’s break down the three foundational pillars of the Transformer: **Positional Encoding**,                   │
│  **Self-Attention**, and **Multi-Head Attention**, using everyday analogies.                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Pillar 1: Positional Encoding (Giving Words an Address)                                                     │
│                                                                                                                 │
│  ### The Problem: Word Soup                                                                                     │
│  Imagine reading a sentence where all the words are shaken up in a bag and dumped onto a table.                 │
│  * *"The dog bit the man."* vs. *"The man bit the dog."*                                                        │
│                                                                                                                 │
│  The words are identical, but the meaning is entirely reversed based on *order*.                                │
│                                                                                                                 │
│  Here is the dirty secret about the core Transformer math: **it suffers from "permutation invariance."** In     │
│  plain English, it doesn't naturally care about the order of words. To the basic math, the sentence above and   │
│  the scrambled version look identical.                                                                          │
│                                                                                                                 │
│  ### The Solution: The Invisible Nametag               

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a beginner-friendly lesson about Transformers.                                                     │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


TASK 1 OUTPUT:
As an AI Researcher on the team, here is the complete, detailed explanation of three fundamental concepts of the Transformer architecture (Positional Encoding, Scaled Dot-Product Self-Attention, and Multi-Head Attention), ready to fulfill the user's criteria.

***

### 1. Positional Encoding

#### The Problem It Solves
Unlike Recurrent Neural Networks (RNNs) that process tokens sequentially (one by one in order), the Transformer architecture processes entire sequences in parallel. While this massively speeds up training, it introduces a critical flaw: **permutation invariance**. 

Because the model looks at all tokens simultaneously, it has no inherent notion of word order. To the self-attention mechanism, the sentences *"The cat chased the mouse"* and *"The mouse chased the cat"* would look mathematically identical. Positional Encoding injects information about the relative or absolute position of the tokens in the sequence into the model.

#### How It Works
Positional

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 5a1e0ac1-64f9-4b1a-81bd-dc6734b376dd                                                                       │
│  Final Output: # Demystifying Transformers: A Beginner-Friendly Guide                                           │
│                                                                                                                 │
│  Welcome to the engine room of modern Artificial Intelligence! If you’ve used tools like ChatGPT, Claude, or    │
│  advanced language translators, you have interacted with a piece of architecture called the **Transformer**.    │
│                                                                                                                 │
│  Introduced in 2017 by Google researchers in the landmark paper *"Attention Is All You Need,"* Transformers     │
│  revolutionized how computers understand human language.                                                        │
│                                                                                                                 │
│  Before Transformers, AI processed text like a conveyor belt—word by word, sequentially. Transformers changed   │
│  everything by looking at an entire sentence all at once. But how do they manage to understand context, word    │
│  order, and multiple meanings simultaneously?                                                                   │
│                                                                                                                 │
│  Let’s break down the three foundational pillars of the Transformer: **Positional Encoding**,                   │
│  **Self-Attention**, and **Multi-Head Attention**, using everyday analogies.                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Pillar 1: Positional Encoding (Giving Words an Address)                                                     │
│                                                                                                                 │
│  ### The Problem: Word Soup                                                                                     │
│  Imagine reading a sentence where all the words are shaken up in a bag and dumped onto a table.                 │
│  * *"The dog bit the man."* vs. *"The man bit the dog."*                                                        │
│                                                                                                                 │
│  The words are identical, but the meaning is entirely reversed based on *order*.                                │
│                                                                                                                 │
│  Here is the dirty secret about the core Transformer math: **it suffers from "permutation invariance."** In     │
│  plain English, it doesn't naturally care about the order of words. To the basic math, the sentence above and   │
│  the scrambled version look identical.                                                                          │
│                                                                                                                 │
│  ### The Solution: The Invisible Nametag              

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## Example 4: Enabling Agent Reasoning in CrewAI

### Objective

Demonstrate how to enable **agent reasoning** in CrewAI, allowing an AI agent to analyze a task, develop a plan, and refine its approach before execution.

The example uses an Artificial Intelligence Lecturer agent to explain the differences between **Breadth-First Search (BFS)** and **A* Search**, including their strategies, completeness, optimality, and memory usage.

Unlike previous examples focused on multi-agent collaboration, this example demonstrates how **a single agent can use reasoning to plan its approach before producing an answer**.

### Implementation

1. **Load environment variables:** Use `load_dotenv()` to load API credentials from the `.env` file.

2. **Initialize the LLM:** Configure a Google Gemini model with:
   - `model="gemini/gemini-3.5-flash-lite"`: Specifies the language model.
   - `temperature=0.1`: Produces more deterministic responses.
   - `max_tokens=1500`: Limits the maximum number of generated tokens per LLM response.

3. **Create an agent with reasoning enabled:** Define an Artificial Intelligence Lecturer responsible for explaining AI concepts.
   - `role`: Defines the agent's specialization.
   - `goal`: Specifies the agent's objective.
   - `backstory`: Provides context about the agent's expertise.
   - `reasoning=True`: Enables task analysis, reflection, and planning before execution.
   - `max_reasoning_attempts=3`: Limits the number of reasoning attempts.
   - `max_iter=5`: Limits the number of iterations during task execution.
   - `verbose=True`: Displays detailed execution information.

4. **Define the task:** Instruct the agent to compare BFS and A*.
   - Explain their search strategies.
   - Compare completeness and optimality.
   - Discuss memory usage.
   - Provide a simple maze example.
   - Present the comparison in a student-friendly format.

5. **Create the Crew:** Configure a Crew containing one agent and one task.
   - `agents=[explanation_agent]`: Specifies the agent responsible for execution.
   - `tasks=[explanation_task]`: Defines the task.
   - `process=Process.sequential`: Executes tasks in their predefined order.
   - `verbose=True`: Displays execution logs.

6. **Execute the Crew:** Use `await crew.kickoff_async()` to start the execution asynchronously in Jupyter Notebook.

7. **Measure execution time:** Use `time.perf_counter()` to calculate the total execution duration, including reasoning and task execution.

8. **Display the final answer:** Use `result.raw` to display the generated explanation.

### Expected Output

The agent generates a student-friendly explanation containing:

**1. Search Strategies**
- BFS explores nodes level by level.
- A* prioritizes nodes using an evaluation function.

**2. Completeness and Optimality**
- Conditions under which BFS finds a solution and guarantees an optimal path.
- Conditions under which A* finds an optimal solution.

**3. Memory Usage**
- BFS stores generated frontier nodes and, in graph search, explored states.
- A* maintains a priority queue and typically tracks explored states and path costs.

**4. Practical Example**
- A simple maze containing an initial state, obstacles, and a goal.
- An explanation of how BFS and A* explore the maze differently.

### Key Concepts
- Agent reasoning
- Task analysis and reflection
- Reasoning plans
- Iterative plan refinement
- Reasoning attempts vs. execution iterations
- Single-agent task execution
- LLM-based planning
- Asynchronous execution
- Execution time measurement

### Important Concept: Agent Reasoning

In CrewAI, agent reasoning allows an agent to **reflect on a task and create a plan before starting its execution**.

When reasoning is enabled, the agent generally follows these steps:

**Step 1 – Analyze the Task**
- Examine the task description.
- Identify the expected output.
- Determine what information and explanations are required.

**Step 2 – Generate a Plan**
- Identify the main concepts to explain.
- Decide how to organize the comparison.
- Determine the steps needed to produce the requested output.

**Step 3 – Evaluate and Refine**
- Evaluate whether the plan is sufficient.
- Identify potential gaps or missing considerations.
- Refine the plan if necessary, within the configured attempt limit.

**Step 4 – Execute the Task**
- Use the resulting plan to guide execution.
- Generate the final explanation.

### Important Concept: Reasoning Attempts vs. Execution Iterations

The example uses two different parameters:

**`max_reasoning_attempts=3`**
- Limits the number of attempts to develop and refine the reasoning plan.
- Applies to the preparation phase before task execution.
- Does not mean that the final answer will be generated three times.

**`max_iter=5`**
- Limits the number of agent execution iterations.
- Applies to the task execution phase.
- Does not mean that the agent must perform exactly five iterations.

These parameters control different stages of the agent's workflow.

### Important Observation: Reasoning vs. No Reasoning

**Without Explicit Reasoning (`reasoning=False`)**
- The agent receives the task.
- It proceeds directly to task execution.
- The LLM generates its response without CrewAI's additional reasoning-planning phase.

**With Explicit Reasoning (`reasoning=True`)**
- The agent receives the task.
- It analyzes the requirements and develops a plan.
- It may refine the plan before execution.
- The resulting plan guides the task execution.

**Important:** Reasoning does not guarantee a more accurate answer. It introduces an additional planning and reflection mechanism that may improve performance on complex tasks.

### Important Observation: Agent Reasoning vs. Hierarchical Execution

**Examples 2 and 3 – Hierarchical Execution**
- A manager coordinates multiple specialized agents.
- The manager assigns tasks to available worker agents.
- The focus is on collaboration, delegation, and coordination.

**Example 4 – Agent Reasoning**
- A single agent performs the task.
- No manager agent is required.
- The agent analyzes and plans its own task execution.
- The focus is on improving how an individual agent approaches a task.

Hierarchical execution and agent reasoning are independent concepts and can also be combined.

### Important Observation: LLM-Based Reasoning vs. Classical AI Planning

In classical AI planning, such as planning with Fast Downward, a planner can search for a sequence of actions that transforms an initial state into a goal state.

For example, A* can search a state space using path costs and heuristic estimates.

**CrewAI Agent Reasoning:**
- Uses an LLM to analyze tasks and formulate a plan.
- Does not automatically construct a formal state-space search problem.
- Does not use A* or another classical search algorithm by default.
- Produces a language-based execution plan rather than a formally verified action sequence.

Therefore, **CrewAI reasoning is different from classical search-based AI planning**.

### Key Takeaway

**Enabling reasoning allows a CrewAI agent to reflect on its assigned task and develop a plan before execution.**

In this example, the Artificial Intelligence Lecturer analyzes how to compare BFS and A*, prepares an approach, and generates a student-friendly explanation.

The example demonstrates that:
- Agent reasoning can be enabled independently of multi-agent collaboration.
- Reasoning attempts and execution iterations are controlled separately.
- Reasoning may improve task organization and execution quality.
- Additional reasoning may increase execution time and LLM usage.

**Important:** Reasoning does not provide the agent with external knowledge. Since no RAG or search tools are configured, the agent relies on the LLM's existing knowledge when generating its explanation.


In [7]:

import time

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

# ==========================================
# 1. LOAD ENVIRONMENT VARIABLES
# ==========================================

load_dotenv()

# ==========================================
# 2. CONFIGURE LLM
# ==========================================

llm = LLM(
    model="gemini/gemini-3.5-flash-lite",
    temperature=0.1,
    max_tokens=1500
)

# ==========================================
# 3. CREATE AGENT WITH REASONING
# ==========================================

explanation_agent = Agent(
    role="Artificial Intelligence Lecturer",
    goal="Explain AI concepts clearly to students",
    backstory=(
        "You are an experienced university lecturer "
        "specializing in Artificial Intelligence."
    ),
    llm=llm,
    reasoning=True,
    max_reasoning_attempts=3,
    max_iter=5,
    verbose=True
)

# ==========================================
# 4. DEFINE TASK
# ==========================================

explanation_task = Task(
    description=(
        "Explain the difference between BFS and A*. "
        "Compare their search strategies, "
        "completeness, optimality, and memory usage. "
        "Provide a simple maze example."
    ),
    expected_output=(
        "A student-friendly explanation containing "
        "a comparison table and a practical example."
    ),
    agent=explanation_agent
)

# ==========================================
# 5. CREATE CREW
# ==========================================

crew = Crew(
    agents=[explanation_agent],
    tasks=[explanation_task],
    process=Process.sequential,
    verbose=True
)

# ==========================================
# 6. EXECUTE - JUPYTER NOTEBOOK
# ==========================================

start = time.perf_counter()

result = await crew.kickoff_async()

print("\n===== FINAL ANSWER =====\n")
print(result.raw)

elapsed = time.perf_counter() - start
print(f"\nExecution time: {elapsed:.2f} seconds")


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.27                                                                                       │
│  Latest version:  1.15.30                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 4cba2eae-ec30-4ad4-a43d-b1b58e31c1cf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Explain the difference between BFS and A*. Compare their search strategies, completeness, optimality,    │
│  and memory usage. Provide a simple maze example.                                                               │
│  ID: d85be757-c8f8-45f2-9863-c8d53453c653                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── 🧠 Reasoning ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Reasoning Started                                                                                              │
│  Attempt: 1                                                                                                     │
│  Status: Thinking...                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: create_reasoning_plan                                                                                    │
│  Args: {'plan': 'Create an execution plan to explain BFS vs A* with definitions, comparison table, maze         │
│  example, and final student-friendly output.', 'ready': True, 'steps': [{'description': 'Define Breadth-...     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Artificial Intelligence Lecturer                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```python                                                                                                      │
│  maze_example = """# Maze Example and Comparison Walkthrough                                                    │
│                                                                                                                 │
│  To see how BFS and A* behave in practice, let's examine a simple grid-based maze example.                      │
│                                                                                                                 │
│  ## The Maze Setup                                                                                              │
│  Consider a 5x5 grid where:                                                                                     │
│  - `S` represents the **Start** node at `(0, 0)`                                                                │
│  - `G` represents the **Goal** node at `(4, 4)`                                                                 │
│  - `#` represents **Obstacles / Walls**                                                                         │
│  - `.` represents **Open path**                                                                                 │
│                                                                                                                 │
│  ```text                                                                                                        │
│  Row 0: [ S ] [ . ] [ # ] [ . ] [ . ]                                                                           │
│  Row 1: [ . ] [ . ] [ # ] [ . ] [ . ]                                                                           │
│  Row 2: [ . ] [ . ] [ . ] [ . ] [ # ]                                                                           │
│  Row 3: [ # ] [ # ] [ # ] [ . ] [ . ]                                                                           │
│  Row 4: [ . ] [ . ] [ . ] [ . ] [ G ]                                                                           │
│  ```                                                                                                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## How BFS Explores the Maze                                                                                   │
│  1. **Strategy**: BFS starts at `S` and expands outward in concentric rings (waves) of equal distance,          │
│  treating every step as having a cost of 1.                                                                     │
│  2. **Behavior**:                                                                                               │
│     - Because it is uninformed, BFS expands nodes equally in all directions (up, down, left, right) toward the  │
│  goal, as well as away from the goal (e.g., hitting the upper walls or exploring dead ends).                    │
│     - It will eventually find the shortest path in terms of steps, but it will have explored a large portion    │
│  of the grid blindly before reaching `G`.              

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Explain the difference between BFS and A*. Compare their search strategies, completeness, optimality,    │
│  and memory usage. Provide a simple maze example.                                                               │
│  Agent: Artificial Intelligence Lecturer                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== FINAL ANSWER =====

```python
maze_example = """# Maze Example and Comparison Walkthrough

To see how BFS and A* behave in practice, let's examine a simple grid-based maze example.

## The Maze Setup
Consider a 5x5 grid where:
- `S` represents the **Start** node at `(0, 0)`
- `G` represents the **Goal** node at `(4, 4)`
- `#` represents **Obstacles / Walls**
- `.` represents **Open path**

```text
Row 0: [ S ] [ . ] [ # ] [ . ] [ . ]
Row 1: [ . ] [ . ] [ # ] [ . ] [ . ]
Row 2: [ . ] [ . ] [ . ] [ . ] [ # ]
Row 3: [ # ] [ # ] [ # ] [ . ] [ . ]
Row 4: [ . ] [ . ] [ . ] [ . ] [ G ]
```

---

## How BFS Explores the Maze
1. **Strategy**: BFS starts at `S` and expands outward in concentric rings (waves) of equal distance, treating every step as having a cost of 1.
2. **Behavior**: 
   - Because it is uninformed, BFS expands nodes equally in all directions (up, down, left, right) toward the goal, as well as away from the goal (e.g., hitting the upper walls or exploring dead ends).
   

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 4cba2eae-ec30-4ad4-a43d-b1b58e31c1cf                                                                       │
│  Final Output: ```python                                                                                        │
│  maze_example = """# Maze Example and Comparison Walkthrough                                                    │
│                                                                                                                 │
│  To see how BFS and A* behave in practice, let's examine a simple grid-based maze example.                      │
│                                                                                                                 │
│  ## The Maze Setup                                                                                              │
│  Consider a 5x5 grid where:                                                                                     │
│  - `S` represents the **Start** node at `(0, 0)`                                                                │
│  - `G` represents the **Goal** node at `(4, 4)`                                                                 │
│  - `#` represents **Obstacles / Walls**                                                                         │
│  - `.` represents **Open path**                                                                                 │
│                                                                                                                 │
│  ```text                                                                                                        │
│  Row 0: [ S ] [ . ] [ # ] [ . ] [ . ]                                                                           │
│  Row 1: [ . ] [ . ] [ # ] [ . ] [ . ]                                                                           │
│  Row 2: [ . ] [ . ] [ . ] [ . ] [ # ]                                                                           │
│  Row 3: [ # ] [ # ] [ # ] [ . ] [ . ]                                                                           │
│  Row 4: [ . ] [ . ] [ . ] [ . ] [ G ]                                                                           │
│  ```                                                                                                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## How BFS Explores the Maze                                                                                   │
│  1. **Strategy**: BFS starts at `S` and expands outward in concentric rings (waves) of equal distance,          │
│  treating every step as having a cost of 1.                                                                     │
│  2. **Behavior**:                                                                                               │
│     - Because it is uninformed, BFS expands nodes equally in all directions (up, down, left, right) toward the  │
│  goal, as well as away from the goal (e.g., hitting the upper walls or exploring dead ends).                    │
│     - It will eventually find the shortest path in terms of steps, but it will have explored a large portion    │
│  of the grid blindly before reaching `G`.             

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## Example 5: Crew-Level Planning in a Multi-Agent System Using CrewAI

### Objective

Demonstrate how to enable **crew-level planning** in CrewAI, allowing a dedicated planning component to analyze the workflow and generate step-by-step execution plans before agents perform their tasks.

The example uses two specialized agents to create an **Artificial Intelligence micro-credential proposal**, including learning outcomes, course topics, and assessment activities.

Unlike Example 4, where reasoning was enabled for an individual agent, this example demonstrates **planning at the Crew level**, coordinating how multiple tasks should be approached.

The system consists of:
- **Agent 1 – Curriculum Specialist:** Designs learning outcomes and course content.
- **Agent 2 – Assessment Specialist:** Creates assessment activities aligned with the learning outcomes.
- **Crew Planner:** Analyzes the overall workflow and generates execution plans for the tasks.

### Implementation

1. **Load environment variables:** Use `load_dotenv()` to load API credentials from the `.env` file.

2. **Initialize the LLM:** Configure a Google Gemini model with:
   - `model="gemini/gemini-3.5-flash-lite"`: Specifies the language model.
   - `temperature=0.1`: Produces more deterministic responses.
   - `max_tokens=1500`: Limits the maximum number of generated tokens per LLM response.

3. **Create two specialized agents:**
   - **Curriculum Specialist:** Responsible for defining learning outcomes and course content.
   - **Assessment Specialist:** Responsible for designing assessment activities and evaluation criteria.
   - `role`: Defines each agent's specialization.
   - `goal`: Specifies the agent's objective.
   - `backstory`: Provides context about the agent's expertise.
   - `allow_delegation=False`: Prevents agents from delegating work.
   - `verbose=True`: Displays detailed execution information.

4. **Define two tasks:**

   **Task 1 – Curriculum Design**
   - Analyze the Artificial Intelligence course outline.
   - Define three measurable learning outcomes.
   - Identify three main course topics.
   - Assign the task to the Curriculum Specialist.

   **Task 2 – Assessment Design**
   - Use the learning outcomes generated by Task 1.
   - Define three assessment activities.
   - Associate each assessment activity with a learning outcome.
   - Specify the corresponding evaluation criteria.
   - Assign the task to the Assessment Specialist.

5. **Define task dependencies:** Use `context=[curriculum_task]` in the assessment task.
   - Makes the curriculum task's output available to the assessment task.
   - Ensures that assessment design can use the previously generated learning outcomes.

6. **Create a Crew with planning enabled:** Configure the Crew using:
   - `agents`: Specifies the two specialized agents.
   - `tasks`: Defines the curriculum and assessment tasks.
   - `process=Process.sequential`: Executes tasks in their predefined order.
   - `planning=True`: Enables crew-level planning.
   - `planning_llm=llm`: Specifies the LLM responsible for generating the execution plans.
   - `verbose=True`: Displays execution information.

7. **Execute the Crew:** Use `await crew.kickoff_async()` to start the execution asynchronously.

   Provide the following inputs:
   - `course_name="Artificial Intelligence"`
   - `course_outline="BFS, DFS, UCS, A*, Constraint Satisfaction Problems, Propositional Logic"`

8. **Measure execution time:** Use `time.perf_counter()` to calculate the total duration, including planning and task execution.

9. **Display the results:** Print the curriculum design, assessment design, and final Crew output.

### Expected Output

The execution produces two main outputs:

**Task 1 – Curriculum Design**
- Three measurable learning outcomes.
- Three main course topics.
- Academic content based on the provided course outline.

**Task 2 – Assessment Design**
- Three assessment activities.
- A mapping between assessment activities and learning outcomes.
- Evaluation criteria for each assessment activity.

**Final Crew Result**
- Returns the output of the final assessment task, which incorporates the learning outcomes generated by Task 1.

### Key Concepts
- Crew-level planning
- Planning LLM
- AgentPlanner
- Multi-agent collaboration
- Sequential task execution
- Task dependencies
- Learning outcome generation
- Assessment alignment
- Asynchronous execution
- Execution time measurement

### Important Concept: Crew-Level Planning

In CrewAI, setting `planning=True` enables an additional planning phase before task execution.

CrewAI uses an internal component called **AgentPlanner** to analyze the Crew configuration and prepare step-by-step plans.

The planning process generally follows these steps:

**Step 1 – Analyze the Workflow**
- Examine the available agents.
- Examine the tasks and their descriptions.
- Identify the expected outputs.
- Consider the available tools and task dependencies.

**Step 2 – Generate Execution Plans**
- Determine how each task should be approached.
- Identify the main steps required to complete the tasks.
- Generate instructions that help the agents perform their work.

**Step 3 – Augment Task Descriptions**
- Add the generated plans to the corresponding task descriptions.
- Provide agents with more detailed execution guidance.

**Step 4 – Execute the Tasks**
- Execute the tasks according to the configured process.
- Agents use the additional planning instructions to guide their work.

### Important Concept: The Planning LLM

The following configuration enables planning:

`planning=True`

The parameter:

`planning_llm=llm`

specifies which language model CrewAI should use for planning.

In this example, the planning component uses the same Google Gemini model as the worker agents.

**Important observations:**
- The planning component is not another worker agent explicitly defined by the developer.
- It analyzes the Crew and prepares task execution plans.
- It does not replace the Curriculum Specialist or Assessment Specialist.
- It does not automatically change the predefined task order or agent assignments.

### Example: Possible Execution Plan

For the Artificial Intelligence micro-credential, the planner might generate the following instructions.

**Plan for Task 1 – Curriculum Design**

1. Analyze the provided course outline.
2. Identify important knowledge and practical skills.
3. Define three measurable learning outcomes.
4. Select three representative course topics.
5. Verify that the topics support the learning outcomes.

**Plan for Task 2 – Assessment Design**

1. Read the learning outcomes generated by Task 1.
2. Identify appropriate assessment methods.
3. Design three assessment activities.
4. Map each activity to its corresponding learning outcome.
5. Define measurable evaluation criteria.

These are illustrative plans. The actual instructions are generated by the planning LLM and may differ.

### Important Observation: Does Planning Change the Task Order?

Consider the following configuration:

`tasks=[curriculum_task, assessment_task]`

The Crew uses:

`process=Process.sequential`

Therefore, the execution order remains:

**Task 1 – Curriculum Design → Task 2 – Assessment Design**

Additionally, Task 2 contains:

`context=[curriculum_task]`

This explicitly makes Task 1's output available to Task 2.

**Enabling planning does not mean that the planner can reverse the tasks or dynamically redesign the workflow.**

The planner generates additional instructions for executing the predefined tasks, while the sequential process determines their execution order.

### Important Observation: Agent Reasoning vs. Crew Planning

**Example 4 – Agent Reasoning**

Configured using:

`reasoning=True`

- Enabled at the individual agent level.
- The agent reflects on its assigned task.
- The agent creates and potentially refines its own execution plan.
- The developer can limit reasoning attempts using `max_reasoning_attempts`.

**Example 5 – Crew Planning**

Configured using:

`planning=True`

- Enabled at the Crew level.
- A dedicated AgentPlanner examines the workflow.
- The planner creates step-by-step instructions for the Crew's tasks.
- The generated instructions are added to task descriptions before execution.
- Multiple agents can benefit from the planning instructions.

**Key Difference:**

Agent reasoning focuses on **how an individual agent should approach its assigned task**.

Crew planning focuses on **preparing execution instructions for the tasks in the overall workflow**.

These features can be used independently or combined.

### Important Observation: Planning vs. Hierarchical Execution

**Hierarchical Execution (Examples 2 and 3)**
- A manager coordinates specialized agents.
- The manager dynamically assigns tasks to available agents.
- The focus is on delegation and coordination.

**Crew Planning (Example 5)**
- A planning component analyzes the workflow before execution.
- Agents remain explicitly assigned to their tasks.
- Tasks execute according to the selected process.
- The focus is on generating better task execution instructions.

Therefore, **planning does not automatically introduce a hierarchical manager or dynamic task delegation**.

### Important Observation: LLM-Based Planning vs. Classical AI Planning

In classical AI planning, algorithms may search a formally defined state space to identify a valid sequence of actions.

For example, Fast Downward can use search algorithms to find a plan that transforms an initial state into a goal state.

**CrewAI Planning:**
- Uses an LLM to generate execution instructions.
- Analyzes task descriptions and agent capabilities.
- Produces natural-language plans.
- Does not automatically use A* or another classical search algorithm.
- Does not formally guarantee that the generated plan is optimal or valid.

Therefore, CrewAI planning is primarily an **LLM-based task preparation mechanism**, rather than classical state-space planning.

### Key Takeaway

**Crew-level planning allows CrewAI to generate structured execution plans before agents perform their assigned tasks.**

In this example, the planner analyzes the curriculum design and assessment design workflow, generates guidance for both tasks, and allows the specialized agents to execute their responsibilities sequentially.

The example demonstrates that:
- Planning can be enabled independently of hierarchical execution.
- The planning LLM generates instructions for the predefined tasks.
- Task dependencies remain explicitly controlled by the developer.
- Planning can improve task organization and consistency.
- Planning introduces additional LLM calls and may increase execution time.

**Important:** The agents do not use external documents or retrieval tools in this example. The generated micro-credential content is based on the supplied course outline and the LLM's existing knowledge.


In [9]:

import time

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

# ==========================================
# 1. LOAD ENVIRONMENT VARIABLES
# ==========================================

load_dotenv()

# ==========================================
# 2. CONFIGURE LLM
# ==========================================

llm = LLM(
    model="gemini/gemini-3.5-flash-lite",
    temperature=0.1,
    max_tokens=1500
)

# ==========================================
# 3. CREATE TWO AGENTS
# ==========================================

curriculum_agent = Agent(
    role="University Curriculum Specialist",
    goal="Design learning outcomes and course content",
    backstory=(
        "You are an experienced university professor "
        "specializing in Artificial Intelligence."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True
)

assessment_agent = Agent(
    role="University Assessment Specialist",
    goal="Design assessments aligned with learning outcomes",
    backstory=(
        "You are an academic expert in student assessment "
        "and university micro-credentials."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True
)

# ==========================================
# 4. DEFINE TASKS
# ==========================================

curriculum_task = Task(
    description=(
        "Design the academic content for a micro-credential "
        "based on the course {course_name}. "
        "The course covers: {course_outline}. "
        "Define 3 measurable learning outcomes and "
        "3 main course topics."
    ),
    expected_output=(
        "A list of 3 learning outcomes and 3 course topics."
    ),
    agent=curriculum_agent
)

assessment_task = Task(
    description=(
        "Using the learning outcomes from the previous task, "
        "design 3 assessment activities. "
        "Explain which learning outcome each activity evaluates."
    ),
    expected_output=(
        "Three assessment activities with their corresponding "
        "learning outcomes and evaluation criteria."
    ),
    agent=assessment_agent,
    context=[curriculum_task]
)

# ==========================================
# 5. CREATE CREW WITH PLANNING
# ==========================================

crew = Crew(
    agents=[
        curriculum_agent,
        assessment_agent
    ],
    tasks=[
        curriculum_task,
        assessment_task
    ],
    process=Process.sequential,

    planning=True,
    planning_llm=llm,

    verbose=True
)

# ==========================================
# 6. EXECUTE CREW (JUPYTER)
# ==========================================

start = time.perf_counter()

try:
    result = await crew.kickoff_async(
        inputs={
            "course_name": "Artificial Intelligence",
            "course_outline": (
                "BFS, DFS, UCS, A*, "
                "Constraint Satisfaction Problems, "
                "Propositional Logic"
            )
        }
    )

    print("\n===== TASK 1: CURRICULUM =====\n")
    print(curriculum_task.output.raw)

    print("\n===== TASK 2: ASSESSMENT =====\n")
    print(assessment_task.output.raw)

    print("\n===== FINAL RESULT =====\n")
    print(result.raw)

except Exception as e:
    print("Error:", e)

finally:
    elapsed = time.perf_counter() - start
    print(f"\nExecution time: {elapsed:.2f} seconds")


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.27                                                                                       │
│  Latest version:  1.15.30                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d75cfdb1-19c4-4f53-a121-ef76c050bf6b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[2026-10-26 10:22:02][INFO]: Planning the crew execution


╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Based on these tasks summary:                                                                            │
│                  Task Number 1 - Design the academic content for a micro-credential based on the course         │
│  Artificial Intelligence. The course covers: BFS, DFS, UCS, A*, Constraint Satisfaction Problems,               │
│  Propositional Logic. Define 3 measurable learning outcomes and 3 main course topics.                           │
│                  "task_description": Design the academic content for a micro-credential based on the course     │
│  Artificial Intelligence. The course covers: BFS, DFS, UCS, A*, Constraint Satisfaction Problems,               │
│  Propositional Logic. Define 3 measurable learning outcomes and 3 main course topics.                           │
│                  "task_expected_output": A list of 3 learning outcomes and 3 course topics.                     │
│                  "agent": University Curriculum Specialist                                                      │
│                  "agent_goal": Design learning outcomes and course content                                      │
│                  "task_tools": []                                                                               │
│                  "agent_tools": "agent has no tools"                                                            │
│                  Task Number 2 - Using the learning outcomes from the previous task, design 3 assessment        │
│  activities. Explain which learning outcome each activity evaluates.                                            │
│                  "task_description": Using the learning outcomes from the previous task, design 3 assessment    │
│  activities. Explain which learning outcome each activity evaluates.                                            │
│                  "task_expected_output": Three assessment activities with their corresponding learning          │
│  outcomes and evaluation criteria.                                                                              │
│                  "agent": University Assessment Specialist                                                      │
│                  "agent_goal": Design assessments aligned with learning outcomes                                │
│                  "task_tools": []                                                                               │
│                  "agent_tools": "agent has no tools"                                                            │
│   Create the most descriptive plan based on the tasks descriptions, tools available, and agents' goals for      │
│  them to execute their goals with perfection.                                                                   │
│  ID: 2f747c17-3a2a-44f7-ac54-4ff1bb38fe5c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on these tasks summary:                                                                            │
│                  Task Number 1 - Design the academic content for a micro-credential based on the course         │
│  Artificial Intelligence. The course covers: BFS, DFS, UCS, A*, Constraint Satisfaction Problems,               │
│  Propositional Logic. Define 3 measurable learning outcomes and 3 main course topics.                           │
│                  "task_description": Design the academic content for a micro-credential based on the course     │
│  Artificial Intelligence. The course covers: BFS, DFS, UCS, A*, Constraint Satisfaction Problems,               │
│  Propositional Logic. Define 3 measurable learning outcomes and 3 main course topics.                           │
│                  "task_expected_output": A list of 3 learning outcomes and 3 course topics.                     │
│                  "agent": University Curriculum Specialist                                                      │
│                  "agent_goal": Design learning outcomes and course content                                      │
│                  "task_tools": []                                                                               │
│                  "agent_tools": "agent has no tools"                                                            │
│                  Task Number 2 - Using the learning outcomes from the previous task, design 3 assessment        │
│  activities. Explain which learning outcome each activity evaluates.                                            │
│                  "task_description": Using the learning outcomes from the previous task, design 3 assessment    │
│  activities. Explain which learning outcome each activity evaluates.                                            │
│                  "task_expected_output": Three assessment activities with their corresponding learning          │
│  outcomes and evaluation criteria.                                                                              │
│                  "agent": University Assessment Specialist                                                      │
│                  "agent_goal": Design assessments aligned with learning outcomes                                │
│                  "task_tools": []                                                                               │
│                  "agent_tools": "agent has no tools"                                                            │
│   Create the most descriptive plan based on the tasks descriptions, tools available, and agents' goals for      │
│  them to execute their goals with perfection.                                                                   │
│  Agent: Task Execution Planner                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: University Assessment Specialist                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  As a University Assessment Specialist, my primary objective is to ensure alignment between measurable          │
│  learning outcomes, student tasks, and rigorous evaluation criteria. Below is the complete assessment design    │
│  for the micro-credential in **Foundations of Classical Artificial Intelligence**, featuring three distinct,    │
│  performance-based assessment activities aligned with the established learning outcomes.                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Assessment 1: Applied Pathfinding Performance Project                                                      │
│                                                                                                                 │
│  * **Target Learning Outcome:**                                                                                 │
│    > *1. Implement and Compare uninformed and informed graph search algorithms (BFS, DFS, UCS, A*) to evaluate  │
│  their time complexity, space complexity, and optimality in solving complex pathfinding problems.*              │
│                                                                                                                 │
│  * **Assessment Activity Description:**                                                                         │
│    Students will be provided with a custom Python framework representing a grid-based logistics and routing     │
│  simulation (a complex, weighted pathfinding problem with obstacles). Students are required to                  │
│  programmatically implement four search algorithms from scratch: Breadth-First Search (BFS), Depth-First        │
│  Search (DFS), Uniform Cost Search (UCS), and A* Search (using two distinct admissible heuristics).             │
│                                                                                                                 │
│    Following implementation, students must execute the algorithms across three standardized map scales (small,  │
│  medium, large with varying obstacle densities) and generate an empirical performance report. The report must   │
│  include a comparative analysis of the algorithms supported by generated data visualizations (plotting nodes    │
│  expanded, path cost, and execution time).                                                                      │
│                                                                                                                 │
│  * **Evaluation Criteria & Rubric:**                                                                            │
│    * **Algorithmic Correctness & Implementation (40%):** Code runs without errors; data structures (priority    │
│  queues, fringe, explored sets) are utilized correctly; algorithms successfully find valid paths where they     │
│  exist.                                                                                                         │
│    * **Optimality & Complexity Analysis (30%):** Accurate empirical measurement and theoretical comparison of   │
│  time/space complexity and solution optimality across a

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the learning outcomes from the previous task, design 3 assessment activities. Explain which        │
│  learning outcome each activity evaluates.Step 1: Retrieve and analyze the 3 measurable learning outcomes       │
│  established in Task 1. Step 2: Design 3 distinct assessment activities (such as projects, quizzes, or          │
│  practical problem-solving tasks) that directly test the competencies described in the learning outcomes. Step  │
│  3: Write a detailed explanation for each assessment activity specifying which exact learning outcome it        │
│  evaluates. Step 4: Define clear evaluation criteria or rubrics for each assessment to measure student          │
│  performance effectively.                                                                                       │
│  Agent: University Assessment Specialist                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== TASK 1: CURRICULUM =====

As a University Curriculum Specialist and Professor of Artificial Intelligence, I have structured the academic content for this micro-credential to ensure both foundational rigor and practical applicability. 

Following the step-by-step curriculum design process, the topics have been synthesized into three cohesive modules, paired with three distinct, measurable learning outcomes utilizing Bloom’s Taxonomy action verbs.

***

### Micro-Credential: Foundations of Classical Artificial Intelligence

#### Part 1: Learning Outcomes
Upon successful completion of this micro-credential, students will be able to:
1. **Implement and Compare** uninformed and informed graph search algorithms (BFS, DFS, UCS, A*) to evaluate their time complexity, space complexity, and optimality in solving complex pathfinding problems.
2. **Formulate and Solve** discrete computational problems as Constraint Satisfaction Problems (CSPs) using systematic search and consistency-enforci

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d75cfdb1-19c4-4f53-a121-ef76c050bf6b                                                                       │
│  Final Output: As a University Assessment Specialist, my primary objective is to ensure alignment between       │
│  measurable learning outcomes, student tasks, and rigorous evaluation criteria. Below is the complete           │
│  assessment design for the micro-credential in **Foundations of Classical Artificial Intelligence**, featuring  │
│  three distinct, performance-based assessment activities aligned with the established learning outcomes.        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Assessment 1: Applied Pathfinding Performance Project                                                      │
│                                                                                                                 │
│  * **Target Learning Outcome:**                                                                                 │
│    > *1. Implement and Compare uninformed and informed graph search algorithms (BFS, DFS, UCS, A*) to evaluate  │
│  their time complexity, space complexity, and optimality in solving complex pathfinding problems.*              │
│                                                                                                                 │
│  * **Assessment Activity Description:**                                                                         │
│    Students will be provided with a custom Python framework representing a grid-based logistics and routing     │
│  simulation (a complex, weighted pathfinding problem with obstacles). Students are required to                  │
│  programmatically implement four search algorithms from scratch: Breadth-First Search (BFS), Depth-First        │
│  Search (DFS), Uniform Cost Search (UCS), and A* Search (using two distinct admissible heuristics).             │
│                                                                                                                 │
│    Following implementation, students must execute the algorithms across three standardized map scales (small,  │
│  medium, large with varying obstacle densities) and generate an empirical performance report. The report must   │
│  include a comparative analysis of the algorithms supported by generated data visualizations (plotting nodes    │
│  expanded, path cost, and execution time).                                                                      │
│                                                                                                                 │
│  * **Evaluation Criteria & Rubric:**                                                                            │
│    * **Algorithmic Correctness & Implementation (40%):** Code runs without errors; data structures (priority    │
│  queues, fringe, explored sets) are utilized correctly; algorithms successfully find valid paths where they     │
│  exist.                                                                                                         │
│    * **Optimality & Complexity Analysis (30%):** Accurate empirical measurement and theoretical comparison of   │
│  time/space complexity and solution optimality across 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯